# 2 — Model comparison across the team

Three members train different model families against the same feature
table and log every run to one shared MLflow server on DagsHub:

| Member | Families |
|---|---|
| Hassan | XGBoost |
| Mudassar | LightGBM, RandomForest, HistGradientBoosting |
| Ifraz | persistence, seasonal naive, rolling mean, Ridge |

This notebook reads that server and builds the comparison table. It is
read-only: it trains nothing and writes nothing.

> **The rule this notebook enforces.** Two models are only comparable
> if they were evaluated on the same data. That sounds obvious and is
> very easy to get wrong when three people fetch from a rolling API on
> three different days. Section 2.2 checks it before showing any
> ranking.

In [ ]:
# Notebooks live in notebooks/, the package lives at the repo root.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({
    "figure.figsize": (11, 4.2),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
print("repo root:", ROOT)

In [ ]:
from src.config import settings
from src.models.tracking import configure

remote = configure()   # points MLflow at DagsHub, or local mlruns/
print("tracking server :", settings.mlflow_tracking_uri if remote else "local mlruns/")
print("experiment      :", settings.mlflow_experiment)
if not remote:
    print("\nNOTE: no MLflow credentials found, so this is reading the local")
    print("store. Set MLFLOW_TRACKING_* in .env to see the team's runs.")

In [ ]:
import mlflow

runs = mlflow.search_runs(experiment_names=[settings.mlflow_experiment])
print(f"{len(runs)} run(s) on the server\n")

cols = {
    "run_id": "run_id",
    "tags.mlflow.runName": "run",
    "tags.member": "member",
    "tags.model_family": "family",
    "tags.dataset_cohort": "cohort",
    "tags.dataset_version": "dataset_md5",
    "tags.quality_gate": "gate",
    "metrics.test_mae": "test_mae",
    "metrics.test_rmse": "test_rmse",
    "metrics.test_r2": "test_r2",
    "metrics.baseline_mae": "baseline_mae",
    "metrics.improvement_pct": "vs_baseline_%",
    "status": "status",
}
present = {k: v for k, v in cols.items() if k in runs.columns}
table = runs[list(present)].rename(columns=present)
table = table.sort_values("test_mae")
table.round(4).to_string(index=False)
table

## 2.2 Comparability check — do these runs share a dataset?

`baseline_mae` is the error of a persistence forecast on a run's own
test split. It depends only on the data, so **two runs that saw the
same split agree on it exactly**. Grouping by it recovers which runs
are mutually comparable, even for runs logged before DVC tracking
existed and therefore carrying no dataset hash.

In [ ]:
key = "cohort" if "cohort" in table.columns else "baseline_mae"
groups = table.groupby(key, dropna=False)

print(f"{groups.ngroups} distinct dataset cohort(s) present\n")
for value, g in groups:
    print(f"cohort baseline_mae={value}  ({len(g)} runs)")
    print(g[["run", "member", "family", "test_mae"]].round(4).to_string(index=False))
    print()

if groups.ngroups > 1:
    print("=" * 66)
    print("WARNING: more than one cohort. Ranking across cohorts is invalid —")
    print("those models were scored on different data. Compare within a cohort")
    print("only, and re-run older models on the current dataset to include them:")
    print("    dvc pull && dvc repro")
    print("=" * 66)

### Why this matters, concretely

If the cells above report two cohorts, look at the XGBoost runs. The
one with the *lower* MAE may well have the *worse* R² — a combination
that is impossible for two models scored on the same test set, and the
signature of a comparison that should never have been made.

The fix is not to pick the better-looking number. It is to pull the
versioned dataset and re-run, so every row shares a cohort:

```bash
dvc pull            # byte-identical data for all three members
dvc repro           # re-runs only what actually changed
```

In [ ]:
# Rank within the cohort the production model belongs to: the largest,
# which is the dataset the team has converged on.
main_cohort = table[key].value_counts().idxmax() if groups.ngroups else None
current = table[table[key] == main_cohort].copy() if main_cohort is not None else table.copy()

print(f"Comparing within cohort: {main_cohort}   ({len(current)} runs)\n")

leaderboard = (
    current.sort_values("test_mae")
    .loc[:, [c for c in ["run", "member", "family", "test_mae", "test_rmse",
                         "test_r2", "vs_baseline_%", "gate"] if c in current.columns]]
    .reset_index(drop=True)
)
leaderboard.round(4)

In [ ]:
plot_df = current.dropna(subset=["test_mae"]).sort_values("test_mae")
if len(plot_df):
    colours = {"xgboost": "#1d4ed8", "lightgbm": "#15803d",
               "baseline": "#94a3b8", "randomforest": "#b45309"}
    bar_colours = [colours.get(str(f).lower(), "#7c3aed") for f in plot_df["family"]]

    fig, ax = plt.subplots(figsize=(11, max(3, 0.52 * len(plot_df))))
    ax.barh(plot_df["run"], plot_df["test_mae"], color=bar_colours, alpha=0.9)

    base = plot_df["baseline_mae"].dropna()
    if len(base):
        ax.axvline(base.iloc[0], color="#dc2626", ls="--", lw=1.4,
                   label=f"persistence baseline ({base.iloc[0]:.2f})")
        ax.legend(loc="lower right")

    for i, v in enumerate(plot_df["test_mae"]):
        ax.text(v + 0.12, i, f"{v:.2f}", va="center", fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("test MAE (ug/m3) — lower is better")
    ax.set_title("Model comparison within one dataset cohort")
    plt.tight_layout()
    plt.show()
else:
    print("No runs with test_mae in this cohort yet.")

## 2.3 Error by forecast horizon

A single MAE hides the thing a user actually cares about: accuracy
decays with how far ahead you ask. Every run logs `mae_h1`, `mae_h6`,
`mae_h12` and `mae_h24`.

In [ ]:
hcols = [c for c in runs.columns if c.startswith("metrics.mae_h")]
if hcols and len(current):
    best_id = current.sort_values("test_mae").iloc[0]["run_id"]
    row = runs[runs["run_id"] == best_id].iloc[0]
    horizons = sorted(int(c.split("_h")[1]) for c in hcols)
    values = [row[f"metrics.mae_h{h}"] for h in horizons]

    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(horizons, values, "o-", color="#1d4ed8", lw=2)
    for h, v in zip(horizons, values):
        ax.annotate(f"{v:.2f}", (h, v), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=9)
    ax.set_xticks(horizons)
    ax.set_xlabel("forecast horizon (hours ahead)")
    ax.set_ylabel("MAE (ug/m3)")
    ax.set_title(f"Error growth with horizon — {row['tags.mlflow.runName']}")
    plt.tight_layout()
    plt.show()

    print(f"+1 h  : {values[0]:.2f} ug/m3")
    print(f"+24 h : {values[-1]:.2f} ug/m3   ({values[-1] / values[0]:.1f}x the 1-hour error)")
else:
    print("No per-horizon metrics logged yet.")

## 2.4 Registry state

Which version is actually being served, and whether it was approved.
Governance requirement: a model reaches Production through a quality
gate and a human promotion, never automatically.

In [ ]:
from mlflow.tracking import MlflowClient

from src.models.registry import MODEL_NAME

client = MlflowClient()
try:
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
except Exception as exc:
    versions = []
    print("registry query failed:", exc)

if versions:
    reg = pd.DataFrame([{
        "version": int(v.version),
        "stage": v.current_stage,
        "run_id": (v.run_id or "")[:8],
        "status": v.status,
    } for v in versions]).sort_values("version", ascending=False)
    print(f"registered model: {MODEL_NAME}")
    print(reg.to_string(index=False))
else:
    print(f"No versions registered under {MODEL_NAME!r} yet.")
    print("Register the best run with:")
    print("    python -m src.models.registry promote --run-id <id> --stage Staging")
    print("    python -m src.models.registry promote-production --version <n>")

## How to add your runs

```bash
conda activate airwatch
dvc pull                       # identical data for everyone
export AIRWATCH_MEMBER=mudassar   # your name — this is the tag the table groups by
python -m src.models.train_lightgbm
```

`src/models/tracking.py` is the only module that talks to MLflow, so
tag and metric names are identical for all three of us by construction
rather than by remembering. Re-run this notebook and your run appears
in the table above — in the same cohort, because you pulled the same
data.